In [9]:
# Import
import os, re, glob, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

In [2]:
# Fixed RandomSeed & Setting Hyperparameter
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 16, 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# Hyperparameters
LOOKBACK = 28
PREDICT  = 7
BATCH_SIZE = 32
EPOCHS = 100
LR = 1e-3
PATIENCE = 8
GRAD_CLIP = 1.0

In [4]:
# Data Load
train = pd.read_csv('./train/train.csv')
train['영업일자'] = pd.to_datetime(train['영업일자'])

In [5]:
# Feature Engineering
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """ df: 단일 시리즈 또는 여러 시리즈 가능 """
    df = df.sort_values(['영업장명_메뉴명', '영업일자']).copy()
    # 요일 주기(sin/cos)
    dow = df['영업일자'].dt.weekday
    df['dow_sin'] = np.sin(2*np.pi*dow/7)
    df['dow_cos'] = np.cos(2*np.pi*dow/7)

    # 시리즈별 롤링/지수평활/추세/제로율
    feats = []
    for k, g in df.groupby('영업장명_메뉴명', group_keys=False):
        s = g['매출수량']

        roll7_mean  = s.shift(1).rolling(7).mean()
        roll14_mean = s.shift(1).rolling(14).mean()
        roll7_std   = s.shift(1).rolling(7).std()

        ewm_a03 = s.shift(1).ewm(alpha=0.3, adjust=False).mean()

        # 최근 14일 선형추세 기울기(shift(1)으로 미래 누수 방지)
        def slope_last_n(x, n=14):
            if len(x) < n: return np.nan
            y = x[-n:]
            X = np.arange(n).reshape(-1,1)
            # 최소제곱 직선 기울기
            b = np.polyfit(np.arange(n), y, 1)[0]
            return b
        trend_14 = s.shift(1).rolling(14).apply(lambda x: slope_last_n(x,14), raw=False)

        # zero rate (28일)
        zero_rate_28 = s.shift(1).rolling(28).apply(lambda x: np.mean(np.array(x)==0), raw=True)

        # 주간 합 비율(최근7일/그 전7일)
        week_sum_recent  = s.shift(1).rolling(7).sum()
        week_sum_prev    = s.shift(8).rolling(7).sum()
        week_sum_ratio   = (week_sum_recent / (week_sum_prev + 1e-6))

        gg = g.copy()
        gg['roll7_mean']   = roll7_mean
        gg['roll14_mean']  = roll14_mean
        gg['roll7_std']    = roll7_std.fillna(0)
        gg['ewm_a03']      = ewm_a03
        gg['trend_14']     = trend_14.fillna(0)
        gg['zero_rate_28'] = zero_rate_28.fillna(0)
        gg['week_sum_ratio'] = week_sum_ratio.replace([np.inf, -np.inf], np.nan).fillna(1.0)
        feats.append(gg)

    out = pd.concat(feats, axis=0).sort_values(['영업장명_메뉴명','영업일자'])
    # 결측/초기구간 안전 처리
    for c in ['roll7_mean', 'roll14_mean', 'ewm_a03']:
        out[c] = out[c].fillna(out[c].median())
    return out

train = add_time_features(train)

# 입력 피처 리스트 (매출은 스케일 후 별도 채널로 사용)
AUX_FEATS = ['dow_sin','dow_cos','roll7_mean','roll14_mean','roll7_std',
             'ewm_a03','trend_14','zero_rate_28','week_sum_ratio']

In [6]:
# Baselines (Naive/SMA/EWMA)
def weekday_naive_from_28days(recent_28_vals):
    arr = np.array(recent_28_vals).reshape(4, 7)  # 4주 x 7일
    return np.mean(arr, axis=0)

def sma7_from_last(vals28):
    # 마지막 7일 평균을 그대로 7일로 복제(레벨 유지)
    m = float(np.mean(vals28[-7:])) if len(vals28)>=7 else float(np.mean(vals28))
    return np.array([m]*7, dtype=float)

def ewma_from_last(vals28, alpha=0.3):
    e = 0.0; w = 0.0
    for v in vals28:
        e = alpha*v + (1-alpha)*e
    # 레벨 예측을 7일로 복제
    return np.array([e]*7, dtype=float)

In [7]:
# LSTM (멀티변수 입력)
class MultiOutputLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, output_dim=7, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        out, _ = self.lstm(x)     # (B, T, H)
        out = out[:, -1, :]       # (B, H)
        out = self.fc(out)        # (B, 7)
        return out

def build_xy_multivar(df_one, scaler_y):
    """단일 시리즈용: 입력 X_t = [y_scaled, AUX_FEATS(norm)], 타깃 y = 다음 7일 y_scaled"""
    g = df_one.sort_values('영업일자')
    y = g['매출수량'].values.reshape(-1,1)
    y_scaled = scaler_y.fit_transform(y)  # 시리즈별 스케일

    # 보조 피처도 간단 정규화(robust하게 평균/표준편차로)
    aux = g[AUX_FEATS].values.astype(np.float32)
    aux_mean = aux.mean(axis=0, keepdims=True); aux_std = aux.std(axis=0, keepdims=True)+1e-6
    aux_scaled = (aux - aux_mean) / aux_std

    series = np.concatenate([y_scaled, aux_scaled], axis=1)  # (N, 1+len(AUX_FEATS))
    X, Y = [], []
    N = len(series)
    for i in range(N - LOOKBACK - PREDICT + 1):
        X.append(series[i:i+LOOKBACK])
        # 타깃은 y_scaled만 (다변량 출력 필요 없음)
        Y.append(y_scaled[i+LOOKBACK:i+LOOKBACK+PREDICT, 0])
    if not X: return None, None, (aux_mean, aux_std)
    return np.asarray(X, np.float32), np.asarray(Y, np.float32), (aux_mean, aux_std)

def train_one_series(df_one):
    df_one = df_one.sort_values('영업일자')
    if len(df_one) < LOOKBACK + PREDICT + 10:
        return None

    scaler_y = MinMaxScaler()
    X, Y, aux_norm = build_xy_multivar(df_one, scaler_y)
    if X is None: return None

    n = len(X); n_val = max(1, int(n*0.2))
    X_tr, Y_tr = X[:n-n_val], Y[:n-n_val]
    X_va, Y_va = X[n-n_val:], Y[n-n_val:]

    tr_loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(Y_tr)),
                           batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(TensorDataset(torch.from_numpy(X_va), torch.from_numpy(Y_va)),
                           batch_size=BATCH_SIZE, shuffle=False)

    model = MultiOutputLSTM(input_dim=X.shape[2], hidden_dim=64, num_layers=2,
                            output_dim=PREDICT, dropout=0.1).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    crit = nn.MSELoss()

    best_va = np.inf; patience = PATIENCE; best_state=None
    for ep in range(EPOCHS):
        model.train()
        for xb, yb in tr_loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            pred = model(xb); loss = crit(pred, yb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        model.eval(); va_losses=[]
        with torch.no_grad():
            for xb, yb in va_loader:
                xb = xb.to(DEVICE); yb = yb.to(DEVICE)
                pred = model(xb); va_losses.append(crit(pred, yb).item())
        va_loss = float(np.mean(va_losses)) if va_losses else np.inf

        if va_loss + 1e-9 < best_va:
            best_va = va_loss; patience = PATIENCE
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            patience -= 1
            if patience <= 0: break

    if best_state is not None:
        model.load_state_dict({k:v.to(DEVICE) for k,v in best_state.items()})

    return {'model': model.eval(), 'scaler_y': scaler_y, 'aux_norm': aux_norm}

def train_lstm(train_df):
    trained = {}
    for sid, g in tqdm(train_df.groupby('영업장명_메뉴명'), desc='Train LSTM+feats'):
        info = train_one_series(g)
        if info is not None:
            trained[sid] = info
    return trained

trained_models = train_lstm(train)

Train LSTM+feats: 100%|██████████| 193/193 [36:19<00:00, 11.29s/it]


In [10]:
# 메타 학습 데이터 만들기 (검증에서 4개 예측 생성)
def generate_meta_training(train_df, trained_models):
    rows = []
    for sid, g in tqdm(train_df.groupby('영업장명_메뉴명'), desc='Build meta data'):
        if sid not in trained_models: continue
        g = g.sort_values('영업일자')
        # 검증 시점들(마지막 20% 윈도우의 시작점들)
        n = len(g)
        max_start = n - LOOKBACK - PREDICT + 1
        if max_start <= 0: continue
        start_idx = int(max_start*0.8)
        idx_list = list(range(start_idx, max_start))  # 검증 영역

        info = trained_models[sid]
        scaler_y = info['scaler_y']; aux_mean, aux_std = info['aux_norm']

        y = g['매출수량'].values.reshape(-1,1)
        y_scaled = scaler_y.fit_transform(y)  # 안전 위해 재적합
        aux = g[AUX_FEATS].values.astype(np.float32)
        aux_scaled = (aux - aux_mean) / (aux_std + 1e-6)
        series = np.concatenate([y_scaled, aux_scaled], axis=1)  # (N, D)

        for i in idx_list:
            # 입력 28일
            Xwin = series[i:i+LOOKBACK]                     # (28, D)
            raw28 = y[i:i+LOOKBACK, 0].reshape(-1)          # 최근 28일 원시값
            # LSTM 예측
            x = torch.from_numpy(Xwin.reshape(1,LOOKBACK,series.shape[1])).float().to(DEVICE)
            with torch.no_grad():
                pred_scaled = info['model'](x).squeeze(0).cpu().numpy()
            # 역변환
            lstm_pred = []
            for h in range(PREDICT):
                z = np.zeros((1,1)); z[0,0] = pred_scaled[h]
                lstm_pred.append(float(scaler_y.inverse_transform(z)[0,0]))
            lstm_pred = np.maximum(lstm_pred, 0).astype(float)

            # Baselines
            if len(raw28) == 28:
                naive = weekday_naive_from_28days(raw28)
                sma7  = sma7_from_last(raw28)
                ewma7 = ewma_from_last(raw28, alpha=0.3)
            else:
                # fallback (거의 없겠지만)
                fill = np.array([raw28[-1]]*7, dtype=float)
                naive = sma7 = ewma7 = fill

            # 정답 7일
            y_true = g['매출수량'].values[i+LOOKBACK:i+LOOKBACK+PREDICT]
            # 수집
            rows.append({
                'sid': sid,
                'naive': naive, 'sma7': sma7, 'ewma': ewma7, 'lstm': lstm_pred,
                'y': y_true
            })
    return rows

meta_rows = generate_meta_training(train, trained_models)

# 호라이즌별 메타 회귀 적합(비음수)
META_MODELS = []  # 길이 7, 각자 LinearRegression(positive=True)
for h in range(PREDICT):
    X_meta = []
    y_meta = []
    for r in meta_rows:
        X_meta.append([r['naive'][h], r['sma7'][h], r['ewma'][h], r['lstm'][h]])
        y_meta.append(r['y'][h])
    if len(X_meta)==0:
        META_MODELS.append(None)
        continue
    X_meta = np.array(X_meta, np.float32)
    y_meta = np.array(y_meta, np.float32)
    m = LinearRegression(positive=True)
    m.fit(X_meta, y_meta)
    META_MODELS.append(m)

Build meta data: 100%|██████████| 193/193 [01:34<00:00,  2.04it/s]


In [11]:
# 추론 (TEST에서 4개 예측 → 메타로 결합)
def predict_one_series_test(g, info):
    """ g: 단일 시리즈의 테스트 28일 프레임(파생 피처 포함) """
    g = g.sort_values('영업일자')
    recent_vals = g['매출수량'].values
    # Baselines
    if len(recent_vals)>=28:
        naive = weekday_naive_from_28days(recent_vals[-28:])
        sma7  = sma7_from_last(recent_vals[-28:])
        ewma7 = ewma_from_last(recent_vals[-28:], alpha=0.3)
    else:
        fill = np.array([recent_vals[-1]]*7, dtype=float)
        naive = sma7 = ewma7 = fill

    # LSTM
    lstm_pred = np.zeros(PREDICT, dtype=float)
    if info is not None and len(g)>=LOOKBACK:
        scaler_y = info['scaler_y']; aux_mean, aux_std = info['aux_norm']
        y = g['매출수량'].values.reshape(-1,1)
        y_scaled = scaler_y.transform(y)  # 학습 시리즈의 스케일러
        aux = g[AUX_FEATS].values.astype(np.float32)
        aux_scaled = (aux - aux_mean) / (aux_std + 1e-6)
        series = np.concatenate([y_scaled, aux_scaled], axis=1)
        Xwin = series[-LOOKBACK:]
        x = torch.from_numpy(Xwin.reshape(1,LOOKBACK,series.shape[1])).float().to(DEVICE)
        with torch.no_grad():
            pred_scaled = info['model'](x).squeeze(0).cpu().numpy()
        tmp=[]
        for h in range(PREDICT):
            z = np.zeros((1,1)); z[0,0] = pred_scaled[h]
            tmp.append(float(scaler_y.inverse_transform(z)[0,0]))
        lstm_pred = np.maximum(tmp, 0.0)

    # 메타 결합
    final = []
    for h in range(PREDICT):
        feats = np.array([[naive[h], sma7[h], ewma7[h], lstm_pred[h]]], dtype=float)
        if META_MODELS[h] is not None:
            v = float(META_MODELS[h].predict(feats)[0])
        else:
            # 메타가 없으면 안전 가중(0.25씩)
            v = float(np.dot(feats[0], np.array([0.25,0.25,0.25,0.25])))
        final.append(max(v, 0.0))
    return np.array(final, dtype=float)

def predict_lstm(test_df, trained_models, test_prefix: str):
    test_df = test_df.copy()
    test_df['영업일자'] = pd.to_datetime(test_df['영업일자'])
    # 파생 피처 부착
    test_df = add_time_features(test_df)

    rows=[]
    for sid, g in test_df.groupby('영업장명_메뉴명'):
        info = trained_models.get(sid, None)
        preds = predict_one_series_test(g, info)
        for i in range(PREDICT):
            rows.append({'영업일자': f"{test_prefix}+{i+1}일",
                         '영업장명_메뉴명': sid,
                         '매출수량': float(preds[i])})
    return pd.DataFrame(rows)

In [12]:
# 실행 & 제출
all_preds=[]
for path in sorted(glob.glob('./test/TEST_*.csv')):
    tdf = pd.read_csv(path)
    tdf['영업일자'] = pd.to_datetime(tdf['영업일자'])
    pref = re.search(r'(TEST_\d+)', os.path.basename(path)).group(1)
    pred_df = predict_lstm(tdf, trained_models, pref)
    all_preds.append(pred_df)

full_pred_df = pd.concat(all_preds, ignore_index=True)

def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    pred_key = dict(((r['영업일자'], r['영업장명_메뉴명']), r['매출수량']) for _,r in pred_df.iterrows())
    out = sample_submission.copy()
    for i in out.index:
        d = out.at[i,'영업일자']
        for col in out.columns[1:]:
            out.at[i,col] = pred_key.get((d, col), 0.0)
    return out

sample_submission = pd.read_csv('./sample_submission.csv')
submission = convert_to_submission_format(full_pred_df, sample_submission)
submission.to_csv('baseline_ensemble_lstm.csv', index=False, encoding='utf-8-sig')